# Basic Decision Tree from Scratch - Regression (Variance)

In this notebook, we are going to build a decision tree from scratch using variance as the search criteria instead of Gini. We will use midpoint from adjacent values as threshold. 

## Dataset

Dataset: This is a makeup dataset that describe how much study hours a student put in and what exam score they can achieve.

Dataset 
| Study Hours  | Exam Score |
| ----- | ----- |
| 1    | 10    |
| 2    | 20    |
| 3    | 30    |
| 4    | 45    |
| 5    | 50   |
| 6    | 55   |
| 7    | 75   |
| 8    | 80   |
| 9    | 80   |
| 10    | 90   |
| 11    | 95   |
| 12    | 93   |
| 13    | 90   |
| 14    | 91   |
| 15    | 88   |

In [1]:
import pandas as pd
import json

In [2]:
df = pd.DataFrame({
    'studied': [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15],
    'score':  [10,20,30,45,50,55,75,80,80,90,95,93,90,91,88]
})

In [3]:
df

,studied,score
0,1,10
1,2,20
2,3,30
3,4,45
4,5,50
5,6,55
6,7,75
7,8,80
8,9,80
9,10,90


## Variance Formula

### Variance Formula

For a node:

$$\text{Variance} = \frac{\sum (y_i - \bar{y})^2}{n}$$


Where:

- $y_i$ = target values
- $\bar{y}$ = mean of target values


**Intuition**

| Case                    | Variance      |
| ----------------------- | ------------- |
| target tightly grouped  | low variance  |
| target widely scattered | high variance |

**A regression tree chooses splits that reduce variance after splitting**

---

### Weighted Variance (for a split)

When you split a node into LEFT and RIGHT:

$$ \text{Weighted Variance} = \frac{N_L}{N} \cdot Var(L) + \frac{N_R}{N} \cdot Var(R)$$

Where:

- $N_L$ = number of samples in left node
- $N_R$ = number of samples in right node
- $N = N_L + N_R$


**Intuition**

Weighted Variance means:

> "How much uncertainty remains after the split, considering how many samples fall into each side?"



### Variance split procedure:
- Sort feature values
- For each adjacent pair:
    - compute midpoint threshold
- For each threshold:
    - split data
    - compute weighted variance
- Choose best threshold

## Variance Optimization Algorithm (Step by Step)

### Step 1 - Sort Features

In [4]:
df_sorted = df.sort_values(by='studied')
df_sorted

,studied,score
0,1,10
1,2,20
2,3,30
3,4,45
4,5,50
5,6,55
6,7,75
7,8,80
8,9,80
9,10,90


### Step 2: Compute Midpoint Threshold For Each Adjacent Pair   

In [5]:
# compute mid points between values of studied
mid_points = (df_sorted['studied'][:-1].values + df_sorted['studied'][1:].values) / 2
mid_points

array([ 1.5,  2.5,  3.5,  4.5,  5.5,  6.5,  7.5,  8.5,  9.5, 10.5, 11.5,
       12.5, 13.5, 14.5])

### Step 3A : Split base on first midpoint threshold

In [6]:
# Split data on first mid point
split_value = mid_points[0]
df_left = df[df['studied'] <= split_value]
df_right = df[df['studied'] > split_value]

print(f"Left split:\n{df_left}\n")
print(f"Right split:\n{df_right}\n")

Left split:
   studied  score
0        1     10

Right split:
    studied  score
1         2     20
2         3     30
3         4     45
4         5     50
5         6     55
6         7     75
7         8     80
8         9     80
9        10     90
10       11     95
11       12     93
12       13     90
13       14     91
14       15     88



In [7]:
df_left['score'].var()

np.float64(nan)

In [8]:
df_left['score'].var(ddof=0)  
# population variance, need ddof=0 to get population variance, 
# default is sample variance (ddof=1) and will return NaN if only one value in the sample
df_right['score'].var(ddof=0)  

print(f"Variance for left split: {df_left['score'].var(ddof=0):.2f}")
print(f"Variance for right split: {df_right['score'].var(ddof=0):.2f}")

Variance for left split: 0.00
Variance for right split: 592.41


### Step 3B: Compute Weighted Variance 

In [9]:
# weighted variance of left and right splits
def weighted_variance(df_left, df_right):
    n_left = len(df_left)
    n_right = len(df_right)
    n_total = n_left + n_right

    var_left = df_left['score'].var(ddof=0)
    var_right = df_right['score'].var(ddof=0)

    #print(f"Variance for left split: {var_left:.2f}")
    #print(f"Variance for right split: {var_right:.2f}")

    weighted_var = (n_left / n_total) * var_left + (n_right / n_total) * var_right

    #print(f"Weighted variance: {weighted_var:.2f}")
    
    return weighted_var

In [10]:
# Compute variance for the left and right splits
weighted_var = weighted_variance(df_left, df_right)

print(f"Weighted variance: {weighted_var:.2f}")

Weighted variance: 552.91


### Step 4: Find best threshold with purest Gini

In [11]:
def best_var_split(df):
    best_variance = float('inf')
    best_split_value = None
    
    for i in range(len(df['studied']) - 1):
        mid_point = (df['studied'][i] + df['studied'][i+1]) / 2
        df_left = df[df['studied'] <= mid_point]
        df_right = df[df['studied'] > mid_point]
        
        current_variance = weighted_variance(df_left, df_right)
        
        if current_variance < best_variance:
            best_variance = current_variance
            best_split_value = mid_point
            
    return best_split_value, best_variance

In [12]:
best_split_value, best_variance = best_var_split(df)
print(f"Best split value: {best_split_value}, Best variance: {best_variance:.2f}")

Best split value: 6.5, Best variance: 131.79


### Create Decision Tree Function

In [13]:
# weighted variance of left and right splits
def weighted_variance(df_left, df_right):
    n_left = len(df_left)
    n_right = len(df_right)
    n_total = n_left + n_right

    var_left = df_left['score'].var(ddof=0)
    var_right = df_right['score'].var(ddof=0)

    #print(f"Variance for left split: {var_left:.2f}")
    #print(f"Variance for right split: {var_right:.2f}")

    weighted_var = (n_left / n_total) * var_left + (n_right / n_total) * var_right

    #print(f"Weighted variance: {weighted_var:.2f}")
    
    return weighted_var

In [14]:
def best_var_split(df):
    best_variance = float('inf')
    best_split_value = None
    
    x = df['studied'].values  # best practice: NumPy

    for i in range(len(x) - 1):
        mid_point = (x[i] + x[i+1]) / 2
        df_left = df[df['studied'] <= mid_point]
        df_right = df[df['studied'] > mid_point]
        
        current_variance = weighted_variance(df_left, df_right)
        print(f"variance: {current_variance:.2f}, threshold: {mid_point}")
        
        if current_variance < best_variance:
            best_variance = current_variance
            best_split_value = mid_point
            
    return best_split_value, best_variance

In [15]:
def build_tree_regression(df):

    # ---- STOP 1: empty node ----
    if df.empty:
        return None

    # ---- STOP 2: too small to split ----
    if len(df) <= 1:
        print("Too small to split, one row left, returning mean score")
        return float(df['score'].mean())
    
    # --- STOP 3: no variance ----
    if df['score'].var(ddof=0) == 0:
        print("No variance in score, returning mean score")
        return float(df['score'].mean())

    # ---- compute median split ----
    best_split_value, best_variance = best_var_split(df)

    print(f"Best split value: {best_split_value}, Best variance: {best_variance:.2f}")

    # ---- split data ----
    left_subset = df[df['studied'] <= best_split_value]
    right_subset = df[df['studied'] > best_split_value]

    print(f"Left subset:\n{left_subset}\n")
    print(f"Right subset:\n{right_subset}\n")
    
    # ---- build node ----
    node = {
        'threshold_studied': float(best_split_value),
        'value': round(float(df['score'].mean()), 2),  # 👈 REGRESSION CHANGE
        'left': None,
        'right': None
    }

    # ---- recursion ----
    node['left'] = build_tree_regression(left_subset)
    node['right'] = build_tree_regression(right_subset)

    return node

### Build Tree 

We build a tree using current data frame.

In [16]:
df

,studied,score
0,1,10
1,2,20
2,3,30
3,4,45
4,5,50
5,6,55
6,7,75
7,8,80
8,9,80
9,10,90


In [17]:
reg = build_tree_regression(df)

variance: 552.91, threshold: 1.5
variance: 375.73, threshold: 2.5
variance: 245.91, threshold: 3.5
variance: 199.55, threshold: 4.5
variance: 160.81, threshold: 5.5
variance: 131.79, threshold: 6.5
variance: 212.62, threshold: 7.5
variance: 297.31, threshold: 8.5
variance: 360.20, threshold: 9.5
variance: 458.78, threshold: 10.5
variance: 562.08, threshold: 11.5
variance: 639.53, threshold: 12.5
variance: 693.98, threshold: 13.5
variance: 743.83, threshold: 14.5
Best split value: 6.5, Best variance: 131.79
Left subset:
   studied  score
0        1     10
1        2     20
2        3     30
3        4     45
4        5     50
5        6     55

Right subset:
    studied  score
6         7     75
7         8     80
8         9     80
9        10     90
10       11     95
11       12     93
12       13     90
13       14     91
14       15     88

variance: 141.67, threshold: 1.5
variance: 66.67, threshold: 2.5
variance: 41.67, threshold: 3.5
variance: 113.54, threshold: 4.5
variance: 186

In [18]:
reg


{'threshold_studied': 6.5,
 'value': 66.13,
 'left': {'threshold_studied': 3.5,
  'value': 35.0,
  'left': {'threshold_studied': 1.5,
   'value': 20.0,
   'left': 10.0,
   'right': {'threshold_studied': 2.5,
    'value': 25.0,
    'left': 20.0,
    'right': 30.0}},
  'right': {'threshold_studied': 4.5,
   'value': 50.0,
   'left': 45.0,
   'right': {'threshold_studied': 5.5,
    'value': 52.5,
    'left': 50.0,
    'right': 55.0}}},
 'right': {'threshold_studied': 9.5,
  'value': 86.89,
  'left': {'threshold_studied': 7.5,
   'value': 78.33,
   'left': 75.0,
   'right': 80.0},
  'right': {'threshold_studied': 12.5,
   'value': 91.17,
   'left': {'threshold_studied': 10.5,
    'value': 92.67,
    'left': 90.0,
    'right': {'threshold_studied': 11.5,
     'value': 94.0,
     'left': 95.0,
     'right': 93.0}},
   'right': {'threshold_studied': 14.5,
    'value': 89.67,
    'left': {'threshold_studied': 13.5,
     'value': 90.5,
     'left': 90.0,
     'right': 91.0},
    'right': 88.0}}

In [19]:
# print tree structure in json format
print(json.dumps(reg, indent=8))


{
        "threshold_studied": 6.5,
        "value": 66.13,
        "left": {
                "threshold_studied": 3.5,
                "value": 35.0,
                "left": {
                        "threshold_studied": 1.5,
                        "value": 20.0,
                        "left": 10.0,
                        "right": {
                                "threshold_studied": 2.5,
                                "value": 25.0,
                                "left": 20.0,
                                "right": 30.0
                        }
                },
                "right": {
                        "threshold_studied": 4.5,
                        "value": 50.0,
                        "left": 45.0,
                        "right": {
                                "threshold_studied": 5.5,
                                "value": 52.5,
                                "left": 50.0,
                                "right": 55.0
                        }
       

In [20]:
# print tree structure with guided lines
def print_tree(node, indent="    "):
    if isinstance(node, dict):
        print(f"{indent}Threshold studied: {node['threshold_studied']}")
        print(f"{indent}├─ Left Node: <={node['threshold_studied']}")
        print_tree(node['left'], indent + "│   ")
        print(f"{indent}└─ Right Node: >{node['threshold_studied']}")
        print_tree(node['right'], indent + "    ")
    else:
        print(f"{indent}Score: {node}")



In [21]:
print_tree(reg)

    Threshold studied: 6.5
    ├─ Left Node: <=6.5
    │   Threshold studied: 3.5
    │   ├─ Left Node: <=3.5
    │   │   Threshold studied: 1.5
    │   │   ├─ Left Node: <=1.5
    │   │   │   Score: 10.0
    │   │   └─ Right Node: >1.5
    │   │       Threshold studied: 2.5
    │   │       ├─ Left Node: <=2.5
    │   │       │   Score: 20.0
    │   │       └─ Right Node: >2.5
    │   │           Score: 30.0
    │   └─ Right Node: >3.5
    │       Threshold studied: 4.5
    │       ├─ Left Node: <=4.5
    │       │   Score: 45.0
    │       └─ Right Node: >4.5
    │           Threshold studied: 5.5
    │           ├─ Left Node: <=5.5
    │           │   Score: 50.0
    │           └─ Right Node: >5.5
    │               Score: 55.0
    └─ Right Node: >6.5
        Threshold studied: 9.5
        ├─ Left Node: <=9.5
        │   Threshold studied: 7.5
        │   ├─ Left Node: <=7.5
        │   │   Score: 75.0
        │   └─ Right Node: >7.5
        │       Score: 80.0
        └─ Right Nod

### Prediction / Inference

To make a prediction we pass the prediction to the tree until we reach a leaf.

In [22]:
# make prediction function
def predict(reg, x):
    node = reg  # start at root of the tree

    # while we are still at a decision node (dictionary)
    while isinstance(node, dict):

        # get the split threshold stored at this node
        threshold = node['threshold_studied']

        # go left if x is smaller or equal to threshold
        if x <= threshold:
            print(f"Going left: {x} <= {threshold}")
            node = node['left']
        # otherwise go right
        else:
            print(f"Going right: {x} > {threshold}")
            node = node['right']

    # when we reach a leaf (not a dict), return the prediction
    return node

In [23]:
number_of_hours_studied = 9

prediction = predict(reg, number_of_hours_studied)
print(f"Prediction for student who studied {number_of_hours_studied} hours: Estimated score: {prediction}")

Going right: 9 > 6.5
Going left: 9 <= 9.5
Going right: 9 > 7.5
Prediction for student who studied 9 hours: Estimated score: 80.0


## Scikit learn Compare

In the following, we will use Scikit-Learn to compare the tree. 

In [24]:
from sklearn.tree import DecisionTreeRegressor

sklearn_reg = DecisionTreeRegressor(criterion='squared_error')

sklearn_reg.fit(df[['studied']], df['score'])

,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 0.24 Poisson deviance criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.For an example of how ``max_depth`` influences the model, see:ref:`sphx_glr_auto_examples_tree_plot_tree_regression.py`.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",None
,"max_leaf_node

In [25]:
# get tree structure in json format
from sklearn.tree import export_text
tree_rules = export_text(sklearn_reg, feature_names=['studied'])
print(tree_rules)

|--- studied <= 6.50
|   |--- studied <= 3.50
|   |   |--- studied <= 1.50
|   |   |   |--- value: [10.00]
|   |   |--- studied >  1.50
|   |   |   |--- studied <= 2.50
|   |   |   |   |--- value: [20.00]
|   |   |   |--- studied >  2.50
|   |   |   |   |--- value: [30.00]
|   |--- studied >  3.50
|   |   |--- studied <= 4.50
|   |   |   |--- value: [45.00]
|   |   |--- studied >  4.50
|   |   |   |--- studied <= 5.50
|   |   |   |   |--- value: [50.00]
|   |   |   |--- studied >  5.50
|   |   |   |   |--- value: [55.00]
|--- studied >  6.50
|   |--- studied <= 9.50
|   |   |--- studied <= 7.50
|   |   |   |--- value: [75.00]
|   |   |--- studied >  7.50
|   |   |   |--- value: [80.00]
|   |--- studied >  9.50
|   |   |--- studied <= 12.50
|   |   |   |--- studied <= 10.50
|   |   |   |   |--- value: [90.00]
|   |   |   |--- studied >  10.50
|   |   |   |   |--- studied <= 11.50
|   |   |   |   |   |--- value: [95.00]
|   |   |   |   |--- studied >  11.50
|   |   |   |   |   |--- value

In [26]:
# make prediction function using sklearn
df_numbers_of_hours_studied = pd.DataFrame({'studied': [number_of_hours_studied]}) 
# need to convert to df as original data during training is in df format

sklearn_prediction = sklearn_reg.predict(df_numbers_of_hours_studied)
print(f"Prediction for student who studied {number_of_hours_studied} hours: Estimated score: {sklearn_prediction}")

Prediction for student who studied 9 hours: Estimated score: [80.]


### Compare Splits and Tree

In [27]:
from sklearn.tree import _tree

def compare_trees(my_node, sk_tree, sk_node_id=0, depth=0):

    # sklearn node info
    sk_feature = sk_tree.feature[sk_node_id]
    sk_threshold = sk_tree.threshold[sk_node_id]

    # leaf check
    if sk_feature == _tree.TREE_UNDEFINED:
        print("  " * depth, "Leaf comparison reached")
        return

    print("  " * depth,
          f"My split vs Sklearn split:")
    print("  " * depth,
          f"Sklearn: X <= {sk_threshold}")

    print("  " * depth,
          f"My:      X <= {my_node['threshold_studied']}")

    # compare similarity
    diff = abs(my_node['threshold_studied'] - sk_threshold)
    print("  " * depth, f"Diff = {diff:.4f}\n")

    # recurse left/right
    compare_trees(my_node['left'], sk_tree, sk_tree.children_left[sk_node_id], depth+1)
    compare_trees(my_node['right'], sk_tree, sk_tree.children_right[sk_node_id], depth+1)

In [28]:
compare_trees(reg, sklearn_reg.tree_)

 My split vs Sklearn split:
 Sklearn: X <= 6.5
 My:      X <= 6.5
 Diff = 0.0000

   My split vs Sklearn split:
   Sklearn: X <= 3.5
   My:      X <= 3.5
   Diff = 0.0000

     My split vs Sklearn split:
     Sklearn: X <= 1.5
     My:      X <= 1.5
     Diff = 0.0000

       Leaf comparison reached
       My split vs Sklearn split:
       Sklearn: X <= 2.5
       My:      X <= 2.5
       Diff = 0.0000

         Leaf comparison reached
         Leaf comparison reached
     My split vs Sklearn split:
     Sklearn: X <= 4.5
     My:      X <= 4.5
     Diff = 0.0000

       Leaf comparison reached
       My split vs Sklearn split:
       Sklearn: X <= 5.5
       My:      X <= 5.5
       Diff = 0.0000

         Leaf comparison reached
         Leaf comparison reached
   My split vs Sklearn split:
   Sklearn: X <= 9.5
   My:      X <= 9.5
   Diff = 0.0000

     My split vs Sklearn split:
     Sklearn: X <= 7.5
     My:      X <= 7.5
     Diff = 0.0000

       Leaf comparison reached
       

## End